# Axe 1 : Conditions de travail

Dans ce notebook, j'étudie les variables liées aux facteurs organisationnelles des employés afin d'évaluer leur impact sur l'attrition.

Ce notebook a pour objectif :
- d'explorer les variables de l'axe,
- de comparer avec l'attrition
- d'explorer des relations internes à l’axe
- de visualiser
- retenir seulement ce qui ressort clairement
---

## 1. Exploration des variables de l’axe
---

### 1.1 Import des librairies & chargement des données
---

In [2]:
# import librairies 

import pandas as pd
import pyarrow
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import os

In [3]:
# chargement du df
df_clean = pd.read_parquet("C:/Users/Kemu/Documents/Formation/Projet Pro Data/2_Projets_Tests/projet-RH_test/data/processed/employees_clean.parquet")
df_clean.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1470 entries, 0 to 1469
Data columns (total 35 columns):
 #   Column                    Non-Null Count  Dtype 
---  ------                    --------------  ----- 
 0   Age                       1470 non-null   int64 
 1   Attrition                 1470 non-null   string
 2   BusinessTravel            1470 non-null   string
 3   DailyRate                 1470 non-null   int64 
 4   Department                1470 non-null   string
 5   DistanceFromHome          1470 non-null   int64 
 6   Education                 1470 non-null   int64 
 7   EducationField            1470 non-null   string
 8   EmployeeCount             1470 non-null   int64 
 9   EmployeeNumber            1470 non-null   int64 
 10  EnvironmentSatisfaction   1470 non-null   int64 
 11  Gender                    1470 non-null   string
 12  HourlyRate                1470 non-null   int64 
 13  JobInvolvement            1470 non-null   int64 
 14  JobLevel                

### 1.2 Fonctions utilitaires

In [ ]:
# Tableau de pourcentage variable univarié

def value_counts_percent(serie,round_n=3):

    """  Calcule la distribution (%) d'une variable catégorielle et retourne un DataFrame"""
    
    return (serie.value_counts(normalize=True).
            round(round_n).
            mul(100).
            reset_index(name="Pourcentage"))

# Tableau de pourcentage de variables bivariés
def groupby_percent(df,columns1,columns2,observed=True):
    """Calcule la distribution (%) de columns2 pour chaque modalité de columns1
    et retourne un DataFrame au format long. """
    return(df.groupby(columns1,observed=observed)[columns2].
           value_counts(normalize=True).
           round(2)
           *100).reset_index(name="Pourcentage")

# Table pivot en pourcentage de variables bivarié
def pivot_percent(df,index,columns):
    """ Crée un tableau pivot (%) : lignes=index, colonnes=columns."""
    df = groupby_percent(df,index,columns)
    return(df.pivot(index=index, columns=columns,values="Pourcentage").fillna(0))




### 1.3 Creation du DataFrame Condition de travail
---
#### Colonnes incluses dans l'axe "Condition de travail"

Cet axe regroupe les caractéristiques organisationnelles :
- BusinessTravel = Structure du poste = leger
- Department = Structure du poste = leger mais si gros impact interresant pour l'attrition
- JobLevel = Structure du poste = leger
- JobRole = Structure du poste = detaillé
- MonthlyIncome = Rémunération = detaillé
- OverTime = structure poste = detaillé
- PercentSalaryHike = Rémuneration = leger
- StockOptionLevel = Rémuneration = leger
- TotalWorkingYears = carriere = detaillé
- TrainingTimesLastYear = evolution = 
- YearsAtCompany = ancienneté = detaillé
- YearsInCurrentRole = carriere = detaillé
- YearsSinceLastPromotion = evolution = detaillé
- YearsWithCurrManager = manager = a voir en lien avec departement
 
donnant un contexte organisationnelle pouvant influencer l'attrition.

In [5]:
# colonnes condition de travail
cols_conditions_travail = ['BusinessTravel', 'Department','JobLevel',"JobRole",'MonthlyIncome','OverTime','PercentSalaryHike','StockOptionLevel',
             "TotalWorkingYears", "TrainingTimesLastYear", "YearsAtCompany","YearsInCurrentRole","YearsSinceLastPromotion","YearsWithCurrManager" ]

# creation df_condition_Trav
df_conditions_travail = df_clean[cols_conditions_travail + ['Attrition']].copy()
df_conditions_travail.head()

,BusinessTravel,Department,JobLevel,JobRole,MonthlyIncome,OverTime,PercentSalaryHike,StockOptionLevel,TotalWorkingYears,TrainingTimesLastYear,YearsAtCompany,YearsInCurrentRole,YearsSinceLastPromotion,YearsWithCurrManager,Attrition
0,Travel_Rarely,Sales,2,Sales Executive,5993,Yes,11,0,8,0,6,4,0,5,Yes
1,Travel_Frequently,Research & Development,2,Research Scientist,5130,No,23,1,10,3,10,7,1,7,No
2,Travel_Rarely,Research & Development,1,Laboratory Technician,2090,Yes,15,0,7,3,0,0,0,0,Yes
3,Travel_Frequently,Research & Development,1,Research Scientist,2909,Yes,11,0,8,3,8,7,3,0,No
4,Travel_Rarely,Research & Development,1,Laboratory Technician,3468,No,12,1,6,3,2,2,2,2,No


#### 1.3.1 Préparation des variables
- Création de tranches
- Regroupements
- Renommage pour la lisibilité




### 1.3  Analyse individuelle des variables 
---
Dans cette section, l’analyse porte sur plusieurs caractéristiques des conditions de travail des employés, afin de mieux comprendre le contexte organisationnel de la population étudiée.

Les variables sont analysées selon différents sous-thèmes :
- la structure du poste,
- la rémunération,
- l’ancienneté et la carrière,
- le management et l’évolution professionnelle.

Certaines variables seront analysées de manière plus approfondie que d’autres.
Cette analyse repose sur une approche **descriptive**, basée sur l’étude des **distributions**, à l’aide de **tableaux de proportions** et de **visualisations graphiques**.

#### 1.3.1 Structure du poste
---
Cette section est consacrée à l’exploration des variables liées à la **structure du poste**, qui regroupent les caractéristiques organisationnelles du travail et certaines conditions de professionnelles des employés.

Les variables étudiées sont les suivantes :

- **Department** : correspond au département de rattachement de l’employé au sein de l’entreprise.
- **JobRole** : désigne le poste occupé par l’employé.
- **JobLevel** : indique le niveau hiérarchique ou le niveau de poste de l’employé.
- **BusinessTravel** : représente la fréquence des déplacements professionnels effectués par l’employé.
- **OverTime** : précise si l’employé effectue ou non des heures supplémentaires.

Dans le cadre de cette analyse :
- **BusinessTravel**, **Department** ,**JobLevel** et **Department** feront l’objet d’une analyse descriptive légère.
- **JobRole** et **OverTime** seront analysées de manière plus détaillée.


Les variables de ce sous-thème sont décrites à travers leurs répartitions afin de mieux comprendre la structure des postes au sein de l’entreprise.

##### Departement

In [16]:
department_count = value_counts_percent(df_conditions_travail["Department"])
department_count

,Department,Pourcentage
0,Research & Development,65.4
1,Sales,30.3
2,Human Resources,4.3


##### Tableau des proportions

##### Histogramme des tranches d'âge




#### 1.3.2 Rémunération
---
Cette section s’intéresse aux variables liées à la **rémunération et aux éléments financiers associés**.

Les variables analysées sont :

- **MonthlyIncome** : correspond au revenu mensuel de l’employé.
- **PercentSalaryHike** : indique le pourcentage d’augmentation de salaire.
- **StockOptionLevel** : représente le niveau d’options d’actions attribuées à l’employé.
 
Ces variables sont analysées de manière descriptive afin de caractériser la distribution des niveaux de rémunération au sein de l’entreprise.

##### Tableau des proportions

#### 

#### Répartition des employés par 


#### 1.3.3 Ancienneté et carrière
---
Cette section porte sur les variables décrivant l’**ancienneté** et le **parcours professionnel** des employés.

Les variables étudiées sont :

- **YearsAtCompany** : représente le nombre d’années passées par l’employé dans l’entreprise.
- **YearsInCurrentRole** : correspond à l’ancienneté dans le poste actuel.
- **TotalWorkingYears** : indique le nombre total d’années d’expérience professionnelle.

L’analyse de ces variables permet de décrire le niveau d’expérience et l’ancienneté des employés au sein de l’organisation.


##### Tableau des proportions

##### Répartition des employés 


#### 1.3.4 Management et évolution
---

Cette dernière section explore les variables relatives au **management** et aux **aspects de l’évolution professionnelle**

Les variables analysées sont :

- **YearsSinceLastPromotion** : indique le nombre d’années écoulées depuis la dernière promotion de l’employé.
- **TrainingTimesLastYear** : correspond au nombre de formations suivies au cours de l’année précédente.
- **YearsWithCurrManager** : représente la durée de collaboration avec le manager actuel.

Ces variables sont décrites afin de mieux caractériser le contexte managérial et les parcours d’évolution au sein de l’entreprise.

##### Tableau des proportions

##### Repartitions des employes par 


## 2. Attrition — Comparaisons simples
---

### 2.1 Introduction
---
Dans cette section, je compare la variable cible, l’attrition, aux variables des conditions de travails analysées précédemment, afin d’identifier d’éventuelles relations entre ces caractéristiques et le départ des employés.


##### Tableau de proportions Attrition Yes / No

In [7]:
attrition = df_vie_perso['Attrition'].value_counts(normalize=True).sort_values(ascending=False).round(3)*100

df_attrition = attrition.reset_index()
df_attrition.columns = ['Attrition', "Pourcentage"]

df_attrition

NameError: name 'df_vie_perso' is not defined

D’après ce tableau, le taux d’attrition global observé dans le jeu de données est d’environ **16 %**, ce qui constitue le niveau de référence pour les analyses suivantes.
Dans les analyses suivantes, l’accent est mis sur le taux d’attrition (Attrition = Yes), les non-départs représentant le complément à 100 %.

### 2.2 Analyse de l’attrition selon les variables des conditions de travail
---
Certaines variables des conditions de travail ont été explorées individuellement mais n’ont pas été approfondies dans l’analyse de l’attrition en raison d’une distribution peu différenciante ou d’un impact limité observé lors de l’exploration initiale.

Les variables sont comparées à partir de leur fréquence, à l’aide de tableaux de proportions et de visualisations graphiques.  
Cette approche permet d’identifier des différences de comportement entre les groupes avant toute analyse approfondie.

#### 2.2.1 Attrition et Department
---

##### Tableau de proportions ­­

##### Graphique — Analyse de la relation entre 

##### Observation descriptive

#### 2.2.2 Attrition et JobRole
---


##### Tableau de proportions

##### Graphique — Analyse de la relation 

##### Observation descriptive

#### 2.2.3 Attrition et MonthlyIncome
---


##### Tableau de proportions

##### Graphique — Analyse de la relation entre 

##### Observation descriptive

#### 2.2.4 Attrition et overtime
---

##### Tableau de proportions

##### Graphique — Analyse de la relation entre 

##### Observation descriptive

#### 2.2.5 Attrition et TotalWorkingYears
---

##### Tableau de proportions

##### Graphique — Analyse de la relation entre 

##### Observation descriptive 

#### 2.2.6 Attrition et TrainingTimesLastYear
---

##### Tableau de proportions

##### Graphique — Analyse de la relation entre 

##### Observation descriptive 

#### 2.2.7 Attrition et YearsAtCompany
---

##### Tableau de proportions

##### Graphique — Analyse de la relation entre 

##### Observation descriptive 

#### 2.2.8 Attrition et YearsInCurrentRole
---

##### Tableau de proportions

##### Graphique — Analyse de la relation entre 

##### Observation descriptive 

#### 2.2.9 Attrition et YearsSinceLastPromotion
---

##### Tableau de proportions

##### Graphique — Analyse de la relation entre 

##### Observation descriptive 

#### 2.2.14 Attrition et YearsWithCurrManager
---

##### Tableau de proportions

##### Graphique — Analyse de la relation entre 

##### Observation descriptive 

Une fois les comparaisons simples réalisées, j’explore les relations internes à l’axe afin d’identifier d’éventuels patterns ou corrélations.

## 3. Relations internes à l’axe condition de travail
----

### 3.1 Introduction

Après avoir analysé les relations entre les variables personnelles et l’attrition, cette section vise à explorer les relations internes entre les variables de l’axe *Condition de travail*, afin d’identifier d’éventuels patterns ou profils récurrents.  
Les variables sont analysées à partir de tableaux croisés et de visualisations graphiques descriptives.

---


### 3.2 Analyse des relations entre variables organisationnelles

#### 3.2.1 Relation entre 
---

##### Tableaux croisés

##### Visualisation

##### Observations descriptives



#### 3.2.2 Relation entre 
---


##### Tableaux croisés

##### Visualisation

##### Observations descriptives


#### 3.2.3 Relation entre 
---


##### Tableau croisé

##### Visualisation

##### Observations descriptives



Après avoir analysé ces trois relations, cette section synthétise les principaux patterns observés.

### 3.3 Synthèse des patterns observés
---
Les analyses des relations internes à l’axe *Condition de travail* mettent en évidence plusieurs patterns structurants :

- **Pattern 1 — Relation entre**  
  
- **Pattern 2 — Relation entre**  
  
- **Pattern 3 — Relation entre**  
  
  Ces patterns permettent de mieux contextualiser les résultats observés précédemment et servent de base à la synthèse globale de l’axe Condition de travail.
  ---

# 4. Résultats clés 
---


---

# 5. Conclusion de l’axe “Condition de travail”
---

